In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.metrics import roc_auc_score
import xgboost as xgb

In [4]:
df_xgb = pd.read_csv('df_enc_xgb.csv')

In [5]:
df_xgb.head()

,age,workclass,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,3.769612,0.210093,-0.420060,0.085599,0.342637,0.103070,0.25586,0.109461,0.0,8.379539,-0.035429,0.245835,0
1,3.183112,0.210093,-0.420060,0.085599,0.484014,0.103070,0.25586,0.109461,0.0,8.379539,-1.817204,0.245835,0
2,2.010110,0.210093,-0.031360,0.085599,0.342637,0.063262,0.12388,0.109461,0.0,8.379539,-0.035429,0.245835,0
3,1.130359,0.210093,-2.363558,0.104209,0.124875,0.063262,0.25586,0.109461,0.0,8.268988,-0.035429,0.245835,0
4,0.177296,0.210093,-0.031360,0.064390,0.342637,0.013220,0.25586,0.109461,0.0,8.268988,-0.035429,0.245835,0


In [6]:
X = df_xgb.drop('income', axis=1)
y = df_xgb['income']
X.shape, y.shape

((32561, 12), (32561,))

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=19)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((22792, 12), (9769, 12), (22792,), (9769,))

In [31]:
df_xgb_val = X_test.copy()
df_xgb_val['target'] = y_test
df_xgb_val.head()

,age,workclass,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,target
31966,-0.849080,0.210093,-0.031360,0.045961,0.041578,0.103070,0.25586,0.109461,0.0,0.0,-0.521368,0.245835,0
29138,-0.335892,0.210093,-0.031360,0.064390,0.134483,0.063262,0.12388,0.109461,0.0,0.0,-0.035429,0.250429,0
26720,1.716860,0.294792,-1.974858,0.104209,0.325116,0.103070,0.12388,0.305737,0.0,0.0,-1.331266,0.245835,0
8866,-0.922393,0.210093,-0.420060,0.045961,0.226641,0.013220,0.25586,0.305737,0.0,0.0,-0.440378,0.245835,0
13002,0.030671,0.210093,-1.586158,0.446848,0.124875,0.448571,0.25586,0.305737,0.0,0.0,1.584366,0.245835,0


In [32]:
df_xgb_val.to_csv('df_xgb_val.csv', index=False)

In [33]:
target = y.value_counts()
res = target[0] / target[1]
print(f'Target disbalance: {res:.3f}')

Target disbalance: 3.153


In [34]:
def objective(trial):

    params = {
        'scale_pos_weight': res,

        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'n_estimators': 2000,
        'early_stopping_rounds': 50,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'eval_metric': 'auc',
        'random_state': 4,
        'verbosity': 0,
        'nthread': -1
    }

    X_t, X_v, y_t, y_v = train_test_split(X_train, y_train, test_size=0.2, random_state=19)

    model = xgb.XGBClassifier(**params)
    model.fit(
    X_t, y_t,
    eval_set=[(X_v, y_v)],
    verbose=False
    )
    y_pred = model.predict_proba(X_v)[:, 1]
    roc_auc = roc_auc_score(y_v, y_pred)

    return roc_auc

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

[I 2026-03-19 17:24:28,408] A new study created in memory with name: no-name-dd65276d-278d-4417-a379-c107bc3a4079
[I 2026-03-19 17:24:28,650] Trial 0 finished with value: 0.9292037871373665 and parameters: {'min_child_weight': 7, 'subsample': 0.9118587087745963, 'colsample_bytree': 0.5776817878622502, 'reg_alpha': 0.023687910225859293, 'reg_lambda': 0.00040052056757459775, 'learning_rate': 0.1823388081995549, 'max_depth': 4}. Best is trial 0 with value: 0.9292037871373665.
[I 2026-03-19 17:24:28,841] Trial 1 finished with value: 0.9292752973564781 and parameters: {'min_child_weight': 3, 'subsample': 0.870249353826001, 'colsample_bytree': 0.5316718877964837, 'reg_alpha': 0.000274871513497801, 'reg_lambda': 0.00356842842968086, 'learning_rate': 0.13636705851770073, 'max_depth': 8}. Best is trial 1 with value: 0.9292752973564781.
[I 2026-03-19 17:24:29,076] Trial 2 finished with value: 0.9295424332324701 and parameters: {'min_child_weight': 6, 'subsample': 0.9607139708447725, 'colsample_b

In [35]:
best_params = study.best_params
best_params

{'min_child_weight': 2,
 'subsample': 0.9608759692780585,
 'colsample_bytree': 0.6343429895835034,
 'reg_alpha': 0.7447034956619789,
 'reg_lambda': 3.1506591797663835e-08,
 'learning_rate': 0.014258742182258484,
 'max_depth': 7}

In [36]:
print(f'ROC AUC score :{study.best_value:.4f}')

ROC AUC score :0.9317


In [37]:
xgb_model = xgb.XGBClassifier(**best_params, random_state=19)

In [38]:
xgb_model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6343429895835034
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [40]:
import joblib
joblib.dump(xgb_model, 'xgb_model.joblib')

['xgb_model.joblib']